# Day 13 — Softmax = a *soft* argmax

**Milestone 2:** the centerpiece. Softmax takes the hard, non-differentiable argmax (day 10) and makes it **soft, smooth, and tunable** — turning scores into a probability distribution we can attend with.

## 1. Review (~5 min)

Recall **day 10** (argmax) and **day 6** (normalize).

In [ ]:
import numpy as np
rng = np.random.default_rng(13)

**R1.** (day 10) index of the largest entry.

In [ ]:
def r_my_argmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    for _ in range(4):
        s = rng.normal(size=int(rng.integers(2, 9)))
        assert fn(s) == int(np.argmax(s))
    print("✅ Review R1 passed")

check_r1(r_my_argmax)

**R2.** (day 6) normalize a vector to unit length.

In [ ]:
def r_normalize(x):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    x = rng.normal(size=5)
    assert np.allclose(np.linalg.norm(np.asarray(fn(x))), 1.0)
    print("✅ Review R2 passed")

check_r2(r_normalize)

## 2. Lecture note (~10 min): softmax

**The itch.** We want argmax's "focus on the biggest" but **smooth and differentiable**, so a model can learn to *shift* its focus.

**The picture.** Softmax converts a vector of scores into a **probability distribution**: all weights positive, summing to 1, with most mass on the larger scores but a little everywhere. A *soft* spotlight instead of a hard one.

**The formula.** Using $e^x$ (day 12) to make things positive, then normalizing (day 6) so they sum to 1:
$$\operatorname{softmax}(s)_i = \frac{e^{s_i}}{\sum_j e^{s_j}}.$$

**Knobs & effect — temperature.** Divide scores by a temperature $T$ first, $\operatorname{softmax}(s/T)$:
- $T\to 0$: the gaps blow up → essentially **one-hot** = argmax (fully sharp).
- $T\to\infty$: the gaps vanish → **uniform** (every option equal).
- $T=1$: the plain version in between.

So softmax is a *dial* between argmax and the uniform average. Because $e^x$ is increasing, softmax also **preserves the ranking** ($\arg\max$ of the weights $=\arg\max$ of the scores).

**In the wild.** Attention weights (how much each token attends to each other token), classification probabilities, RL policies, mixture-of-experts routing.

## 3. Exercises (~15 min)

### Exercise 1 — softmax
Implement $\operatorname{softmax}(s)_i = e^{s_i}/\sum_j e^{s_j}$. (Checks: all positive, sums to 1, matches the formula, preserves the argmax.)

In [ ]:
def softmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(5):
        s = rng.normal(size=int(rng.integers(2, 8)))
        w = np.asarray(fn(s))
        e = np.exp(s - s.max()); oracle = e / e.sum()
        assert np.allclose(w, oracle)
        assert np.all(w > 0) and np.allclose(w.sum(), 1.0)
        assert int(np.argmax(w)) == int(np.argmax(s))
    print("✅ Exercise 1 passed")

check_e1(softmax)

### Exercise 2 — the temperature dial (the effect)
Return $\operatorname{softmax}(s/T)$. Probe the extremes: tiny `T` concentrates almost all weight on the top score; huge `T` spreads weight nearly uniformly.

In [ ]:
def softmax_temperature(s, T):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    s = np.array([1.0, 2.0, 3.0, 4.0])
    cold = np.asarray(fn(s, 0.01))
    assert cold.max() > 0.99                       # nearly one-hot
    assert int(np.argmax(cold)) == int(np.argmax(s))
    hot = np.asarray(fn(s, 100.0))
    assert np.allclose(hot, np.ones(4) / 4, atol=1e-2)  # nearly uniform
    print("✅ Exercise 2 passed")

check_e2(softmax_temperature)

### Exercise 3 — softmax → argmax in the limit
Return softmax at a *very* low temperature and confirm it collapses onto the **one-hot of the argmax** (day 10) — proving softmax really is a soft argmax.

In [ ]:
def hard_limit(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    for _ in range(4):
        s = rng.normal(size=5)
        oh = np.zeros(5); oh[int(np.argmax(s))] = 1.0
        assert np.allclose(np.asarray(fn(s)), oh, atol=1e-6)
    print("✅ Exercise 3 passed")

check_e3(hard_limit)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_my_argmax(s):
    return int(np.argmax(s))

def ref_normalize(x):
    return x / np.linalg.norm(x)

def ref_softmax(s):
    e = np.exp(s - np.max(s))
    return e / e.sum()

def ref_softmax_temperature(s, T):
    return ref_softmax(s / T)

def ref_hard_limit(s):
    return ref_softmax(s / 1e-6)

check_r1(ref_my_argmax)
check_r2(ref_normalize)
check_e1(ref_softmax)
check_e2(ref_softmax_temperature)
check_e3(ref_hard_limit)
print("\nAll reference solutions pass. 🎉  See you on day 14!")